# Кодирование признаков в кибербезопасности

**Рабочая тетрадь студента**  
Практическое занятие — 80 минут. Все способы кодирования соответствуют лекции 6.

## Правила работы

1. `protocol`, `service`, `country`, `alert_rule`, `src_ip`, `domain` — номинальные признаки.
2. `severity`, `confidence`, `asset_criticality` — порядковые признаки; порядок задаётся явно.
3. Словари и статистики кодировщиков рассчитываются только по `train`.
4. Неизвестные категории в `test` обрабатываются по политике, указанной в лекции.
5. Целевые кодировщики используют сглаживание или исключают текущие/будущие события.

In [ ]:
import os, hashlib, math, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
pd.set_option('display.max_columns', None)

DATA_PATH = 'soc_categorical_encoding_lab.csv'
df = pd.read_csv(DATA_PATH, parse_dates=['event_time'])

train = df[df['dataset_part'].eq('train')].copy()
test = df[df['dataset_part'].eq('test')].copy()

print('Размер:', df.shape, '| train:', len(train), '| test:', len(test))
df.head()

## Вспомогательные функции

In [ ]:
def stable_bucket(value, n_components):
    digest = hashlib.sha256(str(value).encode('utf-8')).hexdigest()
    return int(digest, 16) % n_components

def smoothed_target_stats(data, feature, target='is_attack', m=20.0):
    prior = data[target].mean()
    stats = data.groupby(feature)[target].agg(['mean','count'])
    stats['encoded'] = (stats['count']*stats['mean'] + m*prior)/(stats['count']+m)
    return prior, stats

## Задание 1. Аудит категориальных признаков и выбор методов (2 балла)

Выведите размер, типы столбцов, пропуски и дубликаты. Разделите категориальные признаки на номинальные и порядковые. Рассчитайте кардинальность по `train` и составьте таблицу выбора метода для `protocol`, `service`, `alert_rule`, `src_ip`, `country` и `severity` на основе рекомендаций лекции.

In [ ]:
nominal_cols = ['protocol','direction','service','country','alert_rule','src_ip','domain']
ordinal_cols = ['severity','confidence','asset_criticality']
# TODO: аудит, cardinality и таблица feature / cardinality / recommended_method / reason

## Задание 2. Label, One-Hot и Dummy Encoding — 15 минут, 4 балла

Закодируйте `protocol` методом Label Encoding через OrdinalEncoder с кодом −1 для неизвестных категорий. Затем примените OneHotEncoder с `handle_unknown='ignore'` и Dummy Encoding через `pd.get_dummies(..., drop_first=True)`. Сравните число выходных столбцов, интерпретируемость и результат для неизвестной категории `QUIC` в `test`.

In [ ]:
# OrdinalEncoder для nominal protocol: handle_unknown='use_encoded_value', unknown_value=-1
# OneHotEncoder(handle_unknown='ignore', sparse_output=False)
# pd.get_dummies(train['protocol'], drop_first=True, dtype=int)
# Проверьте строки test, где protocol == 'QUIC'

## Задание 3. Binary, Count/Frequency и Feature Hashing (4 балла)

Закодируйте `service` бинарным кодом, рассчитав число битов по числу категорий `train`. Для `service` постройте Count и Frequency Encoding, а неизвестную категорию кодируйте нулём. Для `alert_rule` выполните Feature Hashing в 8 корзин и определите число использованных корзин и коллизий. Сопоставьте размерность и интерпретируемость методов.

In [ ]:
# словарь service -> код; двоичное представление и число битов
# Count/Frequency mapping только по train, неизвестное значение -> 0
# stable_bucket(alert_rule, 8), число корзин и коллизий

## Задание 4. Target, Leave-One-Out и Ordered Target Encoding (5 баллов)

Для `country` рассчитайте сглаженный Target Encoding по `train`, используя общую долю атак и силу сглаживания m=20. Неизвестным категориям присвойте prior. Для `protocol` рассчитайте Leave-One-Out код без текущего события. После сортировки по времени выполните Ordered Target Encoding, используя только предыдущие события той же категории. Сравните защиту методов от утечки целевой переменной.

In [ ]:
# prior и smoothed target statistics для country, m=20
# применить train mapping к test, неизвестные категории -> prior
# LOO(protocol) и Ordered Target Encoding после сортировки event_time

## Задание 5. Кодирование порядковых признаков (5 баллов)

Для `severity` задайте явный порядок `low < medium < high < critical` и выполните Ordinal Encoding. Затем сформируйте One-Hot и Thermometer Encoding. Дополнительно рассчитайте сглаженный Target Encoding и сравните полученный порядок долей атак с естественным порядком категорий. Объясните, какую информацию сохраняет и какое предположение создаёт каждый способ.

In [ ]:
severity_order = ['low','medium','high','critical']
# OrdinalEncoder с явным categories
# One-Hot Encoding
# Thermometer Encoding — пороги medium/high/critical
# smoothed Target Encoding и сравнение порядка

## Самопроверка

Проверьте, что код выполняется сверху вниз и ни один словарь или целевая статистика не рассчитаны по `test`.